In [11]:
import pandas as pd
import numpy as np
import scipy as sp
import sklearn
import importlib

In [16]:
import databox as db
import sample_group_stats as sgst
import joblib


In [ ]:
# Compose raw files for CE encoding from prepared sample groups.
# Run this before book_prepare_ce_dataset

In [243]:
grouplist = sgst.SG._get_group_list()
for gid in grouplist[0:]: # TODO TODO allow all !!!
    print('=============')

    print("Group %s" % gid)
    
    sg0 = sgst.SG(gid)

    cat_s = sg0.ce_data_df.reset_index().set_index('token_id')['rel_role'].str.replace('.*_topic','topic',regex=True).str.replace('_comment','')

    # category of tokens
    # cat_s.sample(5)

    tfdf = sg0.get_tokenfeat_df()
    cat_df = cat_s.reset_index().drop_duplicates().rename(columns={'rel_role':'category'}).set_index('token_id')
    
    # Ensure tokens that can be in comment, without self-reference
    #
    # Polysemous terms may refer to multiple taxonomic levels: bat, frog)
    #
    # a nice example:
    #
    # - print(pdf[pdf.btoken=='Ran-546a_2_frog-10'])
    # - ex.pretty_token('Ran-546a_2_frog-10').t_synt.T

    pdf = sg0.pair_df.reset_index()
    ok_c_tokens = pdf[pdf.btoken!=pdf.rtoken].rtoken.unique()
    catx = cat_df.reset_index()
    catx = catx[(catx.category=='topic') | catx.token_id.isin(ok_c_tokens)]
    cat_df = catx.set_index('token_id')

    # Still ok, e.g. 
    # cat_df[cat_df.index.duplicated(keep=False)].sort_index().head(10)    
   

    
    tfdf_cat = tfdf.set_index('token').merge(cat_df,left_index=True,right_index=True)
    tfdf_cat['sample_grp']=sg0.group_name
    tfdf_cat = tfdf_cat.reset_index().set_index(['token','category','widx'])
    print(tfdf_cat.head(2).T)
    print('=============')


    pq_fn = 'data/q_sep/tokenfeat_cat-'+sg0.group_name+'.pq'
    tfdf_cat.to_parquet(pq_fn)
    print('%s written' % pq_fn)
    print(tfdf_cat.shape)
    print('=============')
    
    # Write context encodings too!
    
    ce = sg0.ce_data_df.reset_index().set_index('token_id').drop(columns=['rel_role'])
    ce = ce[~ce.index.duplicated()].copy()

    # one context encoding per row

    ce_cat = ce.merge(cat_df,left_index=True,right_index=True)
    ce_cat = ce_cat.reset_index().set_index(['token_id','category'])
    pq_fn = 'data/q_sep/context_enc_cat_df-'+sg0.group_name+'.pq'
    ce_cat.to_parquet(pq_fn)

    print('%s written' % pq_fn)
    print(ce_cat.shape)
    print('=============')




Group Accipitridae#family
My URI: Accipitridae#family
My path: data/Accipitridae__family/
My taxons: (28) ['Golden_Eagle#species', 'Harriss_Hawk#species', 'Hen_Harrier#species'] ...
Loaded data/Accipitridae__family//pair_df.pkl
ica_tf.pkl restored
=== get_tokendict Restored SG token dictionary data/Accipitridae__family//token_dict.pkl
==make_tokenfeat_df==
==make_tokenfeat_df done==
token       Acc-0d4b_15_Accipiter-3                      
category                      topic                      
widx                              1                     2
token_form                Accipiter             Accipiter
concept_uri         Accipiter#genus       Accipiter#genus
arcpath                        det>                 amod>
lemma                           the                 genus
upos                            DET                   ADJ
sample_grp     Accipitridae__family  Accipitridae__family
data/q_sep/tokenfeat_cat-Accipitridae__family.pq written
(22452, 6)
data/q_sep/context_enc_c

In [236]:
tfdf_cat.index.droplevel(2).unique()

MultiIndex([( 'Acc-0f45_109_amphibians-16',      'topic'),
            (    'Acc-0f45_109_animals-47', 'taxongroup'),
            (  'Acc-0f45_132_amphibians-3',      'topic'),
            (  'Add-f0f0_105_amphibians-5', 'taxongroup'),
            ('Add-f0f0_105_salamanders-14',      'topic'),
            ( 'Add-f0f0_109_salamander-11',      'topic'),
            (  'Add-f0f0_10_amphibians-12',      'topic'),
            ( 'Add-f0f0_111_salamanders-6',      'topic'),
            ( 'Add-f0f0_112_salamanders-7',      'topic'),
            (   'Aes-3859_41_amphibians-5',      'topic'),
            ...
            ( 'Wet-a585_192_amphibians-33',      'topic'),
            (  'Whi-0286_38_amphibians-37',      'topic'),
            (        'Whi-b4c3_482_toad-8',      'topic'),
            ( 'Whi-b4c3_522_amphibians-34',      'topic'),
            (   'Wil-3cbf_74_amphibians-5', 'taxongroup'),
            (  'Wil-3cbf_74_salamanders-8',      'topic'),
            ( 'Wil-57af_337_amphibians-1

In [240]:
ce_cat.reset_index().category.value_counts()

category
topic         1166
taxongroup     315
adaptation      73
location        63
Name: count, dtype: int64

In [241]:
# demo on multi-word terms

import extools as ex
db.tokentag_df[db.tokentag_df.term.str.contains('nisus')].head(3).T

token_id,Lon-b0af_530_nisus-126,Eur-9431_2_nisus-6,Bir-959d_35_nisus-20
document,Long_eared_owl-en-b0af,Eurasian_sparrowhawk-en-9431,Bird_of_prey-en-959d
sent_id,530,2,35
word_id,126,6,20
concept_uri,Eurasian_Sparrowhawk#species,Eurasian_Sparrowhawk#species,Eurasian_Sparrowhawk#species
term_id,2172,2172,2172
coverage,100,100,100
score,87,87,87
token_form,nisus,nisus,nisus
found_parts,2,2,2
req_parts,2,2,2


In [114]:
# db.tokentag_df.loc['Acc-0d4b_16_Accipiter-9']
ex.pretty_token('Acc-0d4b_16_nisus-10').t_arcs_ex_term


,arcpath,lemma,upos,widx
291,appos<nsubj>det>,the,DET,1
292,appos<nsubj>compound>,type,NOUN,2
290,appos<nsubj>,species,NOUN,3
293,appos<cop>,be,AUX,4
294,appos<det>,the,DET,5
295,appos<amod>,Eurasian,ADJ,6
289,appos<,sparrowhawk,NOUN,7


In [117]:
ex.pretty_token('Acc-0d4b_16_nisus-10').t_synt.T

widx,1,2,3,4,5,6,7,8,9,10,11,12
form,The,type,species,is,the,Eurasian,sparrowhawk,(,Accipiter,nisus,),.
arcpath,appos<nsubj>det>,appos<nsubj>compound>,appos<nsubj>,appos<cop>,appos<det>,appos<amod>,appos<,,compound>,,,
